In [1]:
import sqlite3
import pandas as pd


conn = sqlite3.connect("olympics.db")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())
import os
print(os.getcwd())

[]
c:\Users\nicho\AppData\Local\Programs\Microsoft VS Code


In [2]:

query = """
SELECT country, COUNT(*) as medals
FROM results
WHERE medal IS NOT NULL
GROUP BY country
ORDER BY medals DESC
"""
df = pd.read_sql(query, conn)
print(df.head())

DatabaseError: Execution failed on sql '
SELECT country, COUNT(*) as medals
FROM results
WHERE medal IS NOT NULL
GROUP BY country
ORDER BY medals DESC
': no such table: results

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv
import time

URL = "https://www.olympics.com/en/milano-cortina-2026/results/general-reports?discipline=IHO"

driver = webdriver.Chrome()
driver.get(URL)
wait = WebDriverWait(driver, 10)

# Keep clicking "Load more" until it disappears
while True:
    try:
        load_more = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'Load')]"))
        )
        load_more.click()
        time.sleep(1)
    except:
        break  # No more button

rows = driver.find_elements(By.CSS_SELECTOR, "[data-testid='report-row']")

data = []
for row in rows:
    name = row.find_element(By.CSS_SELECTOR, ".report-name").text
    version = row.find_element(By.CSS_SELECTOR, ".report-version").text
    link = row.find_element(By.TAG_NAME, "a").get_attribute("href")
    data.append([name, version, link])

driver.quit()

# Write CSV
with open("olympics_reports.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Report Name", "Version", "Link"])
    writer.writerows(data)



In [4]:
df = pd.read_csv("olympics_reports.csv")
print(df.head())

Empty DataFrame
Columns: [Report Name, Version, Link]
Index: []


In [ ]:
import requests
from bs4 import BeautifulSoup
import tabula
import os

URL = "https://www.olympics.com/en/milano-cortina-2026/results/general-reports?discipline=IHO"

# Step 1: Fetch page
html = requests.get(URL).text
soup = BeautifulSoup(html, "html.parser")

# Step 2: Identify the three PDFs we want
TARGET_NAMES = [
    "Results Book",
    "Medal Standings",
    "Medallists by Event"
]

pdf_links = {}

for link in soup.find_all("a"):
    text = link.get_text(strip=True)
    for target in TARGET_NAMES:
        if target.lower() in text.lower():
            pdf_links[target] = link["href"]

print("Found PDF links:")
for k, v in pdf_links.items():
    print(k, "→", v)

# Step 3: Download PDFs
os.makedirs("pdfs", exist_ok=True)

for name, url in pdf_links.items():
    filename = f"pdfs/{name.replace(' ', '_')}.pdf"
    print("Downloading:", filename)
    r = requests.get(url)
    with open(filename, "wb") as f:
        f.write(r.content)

# Step 4: Extract tables → CSV
os.makedirs("csv_output", exist_ok=True)

for name in TARGET_NAMES:
    pdf_path = f"pdfs/{name.replace(' ', '_')}.pdf"
    print("Extracting tables from:", pdf_path)

    tables = tabula.read_pdf(pdf_path, pages="all", multiple_tables=True)

    for i, table in enumerate(tables):
        csv_path = f"csv_output/{name.replace(' ', '_')}_table_{i+1}.csv"
        table.to_csv(csv_path, index=False)
        print("Saved:", csv_path)


In [ ]:
pdf_url = "https://wmr-static-assets.scd.dgplatform.net/wmr/static/_PDF/OWG2026/IHO/Results_Book_IHO.pdf"
import tabula
# Download + extract all tables from all pages
tables = tabula.read_pdf(pdf_url, pages="all", multiple_tables=True)

# Save each table as its own CSV
for i, table in enumerate(tables):
    table.to_csv(f"results_book_table_{i+1}.csv", index=False)

print("Done!")



Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'
Got stderr: Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Apr 13, 2026 8:19:33 PM org.apache.pdfbox.pdmodel.font.P

Done!


In [3]:
import requests
url = "https://www.olympics.com/en/api/v1/medalists/owg2026/IHO"
data = requests.get(url).json()

rows = []

for event in data["medalists"]:
    event_name = event["eventName"]
    for medal in event["medals"]:
        rows.append({
            "event": event_name,
            "medal": medal["medalType"],
            "athlete": medal["athleteName"],
            "noc": medal["countryCode"]
        })

df = pd.DataFrame(rows)
df.to_csv("medalists_by_event.csv", index=False)

print("Saved medalists_by_event.csv")

ConnectionError: ('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None))

In [ ]:
df